In [1]:
# Initialize config


from dataclasses import dataclass

@dataclass
class ModelParameters:
    vocab_size: int  # total token in vocabulary
    d_model:int   # embedding dimension
    max_seq_len:int  # context window
    num_layers: int # here layers mean decoder block
    num_heads: int # no. of attention head

In [2]:
model_params = ModelParameters(
    vocab_size=44,
    d_model=32,
    max_seq_len=5,
    num_layers=1,
    num_heads=4
)

In [3]:
model_params

ModelParameters(vocab_size=44, d_model=32, max_seq_len=5, num_layers=1, num_heads=4)

In [4]:
from torch import nn
import torch

In [5]:
class EmbeddingLayer(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()
        self.token_embeddings=nn.Embedding(model_params.vocab_size,model_params.d_model)
        self.pos_embeddings = nn.Embedding(model_params.max_seq_len,model_params.d_model)

    def forward(self,token_ids):
        pos_ids = torch.arange(token_ids.shape[1])
        embedding = self.token_embeddings(token_ids) + self.pos_embeddings(pos_ids)
        return embedding


In [6]:
embedding_layer = EmbeddingLayer(model_params)

In [7]:
embedding_layer

EmbeddingLayer(
  (token_embeddings): Embedding(44, 32)
  (pos_embeddings): Embedding(5, 32)
)

In [8]:
x = torch.randint(0,model_params.vocab_size,(2,5))
x

tensor([[19, 16,  6, 41, 18],
        [41, 16, 31, 42,  2]])

In [9]:
output = embedding_layer(x)

In [10]:
output.shape

torch.Size([2, 5, 32])

In [11]:
w_q= nn.Linear(model_params.d_model,model_params.d_model, bias=False)
w_k = nn.Linear(model_params.d_model,model_params.d_model, bias=False)
w_v = nn.Linear(model_params.d_model, model_params.d_model, bias=False)

In [12]:
w_q

Linear(in_features=32, out_features=32, bias=False)

In [13]:
Q =  w_q(output)
K = w_k(output)
V =  w_v(output)

In [15]:
Q.shape

torch.Size([2, 5, 32])

In [16]:
scores = Q @ K.transpose(-2,-1)

In [18]:
scores.shape

torch.Size([2, 5, 5])

In [20]:
casual_mask = torch.tril(torch.ones(scores.shape[1],scores.shape[2]))
masked_scores = scores.masked_fill(casual_mask==0,float('-inf'))
masked_scores

tensor([[[ 0.2267,    -inf,    -inf,    -inf,    -inf],
         [ 1.7421,  3.2649,    -inf,    -inf,    -inf],
         [ 4.1626, -0.6430,  0.8103,    -inf,    -inf],
         [-3.4835, -4.1209,  2.1424, -3.5122,    -inf],
         [-3.7464,  4.2837, -1.7851,  2.9654, -3.5263]],

        [[-4.0757,    -inf,    -inf,    -inf,    -inf],
         [-0.9161,  3.2649,    -inf,    -inf,    -inf],
         [-0.9147, -7.0167, -0.6245,    -inf,    -inf],
         [-1.5667,  0.5375,  1.6029, -1.7941,    -inf],
         [ 3.4422,  4.2667,  4.9102, -0.0907, -3.4804]]],
       grad_fn=<MaskedFillBackward0>)

In [21]:
attention_weights = torch.softmax(masked_scores,dim = -1)

In [23]:
attention_weights.shape

torch.Size([2, 5, 5])

In [24]:
context_vector = attention_weights @ V

In [26]:
context_vector.shape

torch.Size([2, 5, 32])

In [29]:
class CasualSelfAttention(nn.Module):
    def __init__(self, model_params):
        super().__init__()
        self.w_q= nn.Linear(model_params.d_model,model_params.d_model, bias=False)
        self.w_k = nn.Linear(model_params.d_model,model_params.d_model, bias=False)
        self.w_v = nn.Linear(model_params.d_model, model_params.d_model, bias=False)
        self.register_buffer(
                    "casual_mask", 
                    torch.tril(
                        torch.ones(
                            model_params.max_seq_len, 
                            model_params.max_seq_len
                            )
                        )
                )

    def forward(self,x):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        scores = (q @ k.transpose(-2,-1)) /(model_params.d_model ** 0.5)
        mask = self.casual_mask[:x.shape[1],:x.shape[1]]
        masked_scores = scores.masked_fill(mask==0,float("-inf"))
        attention_scores = torch.softmax(masked_scores,dim=-1)
        context_embedding = attention_scores @ v
        return context_embedding




In [30]:
attention_layer = CasualSelfAttention(model_params)

In [31]:
context_out = attention_layer(output)
context_out.shape

torch.Size([2, 5, 32])